# 04.10 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/10-regularisation/exercises.ipynb)

## Exercise 1: Ridge by hand

On the five-row table ($S_{xy} = 500$, $S_{xx} = 40$, $\bar x = 6$, $\bar y = 100$), compute the Ridge slope and intercept for α = 20 and α = 160. What happens to the line as α grows without limit?

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

x = np.array([2, 4, 6, 8, 10.])            # visits last month
y = np.array([50, 80, 90, 130, 150.])      # spend next month, £
X5 = x.reshape(-1, 1)

In [ ]:
# your code here

## Exercise 2: forgetting to scale

Fit `LassoCV` on the noisy 20-customer table **without** `StandardScaler`. Compare the coefficients with the scaled version. Which features are treated unfairly?

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

cust20 = pd.read_csv(data_path("shopwave_customers_small.csv"))
X20 = cust20[["visits_last_month", "items_last_month", "days_since_last"]].copy()
rng = np.random.default_rng(7)
for i in range(1, 9):                       # eight columns of pure noise
    X20[f"noise_{i}"] = rng.normal(size=len(X20))
y20 = cust20["spend_next_month"]
X20.round(2).head()

In [ ]:
# your code here

## Exercise 3: more noise columns

Add 30 noise columns instead of 8. Compare CV MAE for least squares, RidgeCV and LassoCV. Which one suffers most?

In [ ]:
# your code here

## Exercise 4: choose C by CV

On the churn data, use `LogisticRegressionCV(Cs=20, penalty="l1", solver="liblinear", scoring="roc_auc")` inside a pipeline with a scaler. What C does it choose and which features does it keep? Score it once on the test set.

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")                       # ~1 s after the first run
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])
features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}")

cust["avg_basket"] = cust["spend"] / cust["orders"]
cust["orders_per_month"] = cust["orders"] / (cust["tenure_days"] / 30).clip(lower=1)
cust["log_spend"] = np.log1p(cust["spend"])
country = past.groupby("Customer ID")["Country"].agg(lambda s: s.mode()[0])
cust["country"] = country.reindex(cust.index)
cust["is_uk"] = (cust["country"] == "United Kingdom").astype(int)
weekend = past.assign(wk=past["InvoiceDate"].dt.dayofweek >= 5).groupby("Customer ID")["wk"].mean()
cust["share_weekend"] = weekend.reindex(cust.index)
autumn = past[past["InvoiceDate"] >= "2011-06-01"].groupby("Customer ID")["Invoice"].nunique()
cust["orders_last_3m"] = autumn.reindex(cust.index).fillna(0)

churn_cols = ["orders", "spend", "products", "tenure_days", "days_since_last", "avg_basket",
              "orders_per_month", "orders_last_3m", "is_uk", "share_weekend"]
Xc = cust[churn_cols].copy()
skewed = ["orders", "spend", "products", "avg_basket", "orders_per_month", "orders_last_3m"]
Xc[skewed] = np.log1p(Xc[skewed])
from sklearn.model_selection import train_test_split
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, cust["churned"], test_size=0.25,
                                                        random_state=42, stratify=cust["churned"])

In [ ]:
# your code here